In [2]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

C:\Users\1234\AppData\Local\Temp\ipykernel_21376\4159541439.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


In [7]:
from pathlib import Path

In [8]:
load_dotenv(override=True)

True

In [9]:
openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [10]:
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set. Please set it in your .env file.")
else:
    print(f"OPENAI_API_KEY is set, starts with {openai_api_key[:9]}")

OPENAI_API_KEY is set, starts with sk-proj-z


In [11]:
knowledgePath = "knowledge-base/**/*.md"
files = glob.glob(knowledgePath, recursive=True)
print(f"Found {len(files)} files in the knowledge base.")

full_knowledge = ""
for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        full_knowledge += f.read()
        full_knowledge += "\n\n"  # Add a separator between files

print(f"Total length of knowledge base: {len(full_knowledge):,} characters.")

Found 76 files in the knowledge base.
Total length of knowledge base: 304,434 characters.


In [12]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens= encoding.encode(full_knowledge)
token_count = len(tokens)
print(f"total tokens for {MODEL} are {token_count:,} tokens.")


total tokens for gpt-4.1-nano are 63,555 tokens.


In [13]:
folders = glob.glob("knowledge-base/*")
documents = []
for folder in folders:
    doc_type= os.path.basename(folder)
    # print(doc_type)
    loader = DirectoryLoader(folder,glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    folder_docs = loader.load()
    # print(folder_docs)
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)
# print(documents)
print(f"Loaded {len(documents)} documents from the knowledge base.")

Loaded 76 documents from the knowledge base.


In [26]:
documents[0]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content="# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a

In [14]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")


Split into 413 chunks.


In [28]:
chunks[9]

Document(metadata={'source': 'knowledge-base\\company\\careers.md', 'doc_type': 'company'}, page_content='### Operations & Support\n\n**Technical Support Specialist** - Remote\n- Provide tier 2/3 technical support to clients\n- Troubleshoot platform issues\n- Create documentation and knowledge base articles\n- 2+ years technical support experience\n\n**HR Business Partner** - San Francisco, CA\n- Partner with leadership on people strategy\n- Support talent development and retention\n- Drive culture and engagement initiatives\n- 4+ years HR experience, tech industry preferred\n\n## How to Apply\n\nVisit our careers portal at careers.insurellm.com or send your resume to jobs@insurellm.com. Please include the position title in your subject line.\n\nInsurellm is an equal opportunity employer. We celebrate diversity and are committed to creating an inclusive environment for all employees.')

In [ ]:
import sentence_transformers

In [16]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorStore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vector store created and persisted in {vectorStore._collection.count()}.")

Vector store created and persisted in 413.


In [17]:
collection = vectorStore._collection
count = collection.count()
sample_embedding = collection.get(limit=1,include=["embeddings"])["embeddings"][0]
dimensions=len(sample_embedding)
print(count)
print(dimensions)
print(sample_embedding)

413
3072
[-0.01309204  0.03399658 -0.0216217  ... -0.0069046   0.00335884
  0.0002923 ]


In [18]:
result = collection.get(include=['embeddings','documents','metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue','green','red','orange'][['products','employees','contracts', 'company'].index(t)] for t in doc_types]


print (doc_types)
# print(colors)

['company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'company', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contrac

In [19]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
print(reduced_vectors)
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5,color=colors, opacity=0.8),
    text =[f"Type: {t}<br>Text: {d[:100]}" for t,d in zip(doc_types, documents)],
    hoverinfo='text'
)])
fig.update_layout(title='t-SNE Visualization of Document Embeddings', xaxis_title='t-SNE 1', yaxis_title='t-SNE 2',
width=800, height=600,
margin=dict(l=40, r=40, t=40, b=40)
)

[[-13.562652   -12.612956     2.5742166 ]
 [-16.544527   -10.431002    -1.097694  ]
 [-18.144629   -15.147036    -1.8142924 ]
 ...
 [  8.47308    -13.708506    -2.9731374 ]
 [ -1.8817486  -22.840382    -0.24151957]
 [ -7.5268097  -17.973452     0.1493216 ]]


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hoverinfo': 'text',
              'marker': {'color': [orange, orange, orange, ..., blue, blue, blue],
                         'opacity': 0.8,
                         'size': 5},
              'mode': 'markers',
              'text': [Type: company<br>Text: # About Insurellm  Insurellm was
                       founded by Avery Lancaster in 2015 as an insurance tech
                       startup des, Type: company<br>Text: However, the company
                       underwent a strategic restructuring in 2022-2023 to focus on
                       profitability and , Type: company<br>Text: - **Bizllm**
                       quickly gained traction with 7 commercial insurance
                       contracts, including regional carri, ..., Type:
                       products<br>Text: ### Regulatory Compliance Tools Rellm
                       includes built-in compliance tracking features to help
                       organiz, Type: products<br>Text: Join the growing number of
                       organizations leveraging Rellm to enhance their reinsurance
                       processes whi, Type: products<br>Text: Experience the future
                       of reinsurance with Rellm, where innovation meets
                       reliability. Let Insurellm h],
              'type': 'scatter3d',
              'x': {'bdata': ('nwBZwTFbhMEzKJHBskFVwa+rKsHbJk' ... 'xAb5vSwBNa7sC8kQdBI93wv6Db8MA='),
                    'dtype': 'f4'},
              'y': {'bdata': ('q85JwWLlJsFCWnLB4XzPwFVi+D9f62' ... 'XB1amZwcN4tcEKVlvBGrm2waHJj8E='),
                    'dtype': 'f4'},
              'z': {'bdata': ('978kQD2BjL+8Oui/hQwcQKXXq0DiCQ' ... 'DBZAJnwOdaGcDiRz7A6FB3vsPnGD4='),
                    'dtype': 'f4'}}],
    'layout': {'height': 600,
               'margin': {'b': 40, 'l': 40, 'r': 40, 't': 40},
               'template': '...',
               'title': {'text': 't-SNE Visualization of Document Embeddings'},
               'width': 800,
               'xaxis': {'title': {'text': 't-SNE 1'}},
               'yaxis': {'title': {'text': 't-SNE 2'}}}
})